# BERT-Base uncased — DIMER fill-mask and sentence-embedding tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bert-masked-lm-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bert-masked-lm-pipeline/blob/main/tutorials/bert_masked_lm_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google--bert%2Fbert--base--uncased-ffcc4d?style=flat)](https://huggingface.co/google-bert/bert-base-uncased) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fbert-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/bert) [![arXiv](https://img.shields.io/badge/arXiv-1810.04805-b31b1b.svg)](https://arxiv.org/abs/1810.04805)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** masked-language modelling (fill-mask: ranked candidates for one `[MASK]`) and sentence embeddings (768-d, CLS or mean pooled, L2-normalised) using the pinned BERT-Base uncased weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/bert_masked_lm_pipeline/pipeline.py` at revision `db771a57d548`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `86b5e0934494bd15c9632b12f734a8a67f723594` (~441 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/1); edit the repository and regenerate rather than editing cells.

At inference the WordPiece tokenizer lower-cases the text and one forward pass of the 12-layer bidirectional encoder runs; `fill_mask` reads the output-vocabulary logits at the single `[MASK]` position through the masked-LM head and ranks them by a softmax over the 30,522-token vocabulary, while `embed` takes the encoder's last hidden states and pools them to one 768-d vector per text (the `[CLS]` position, or the attention-masked mean) and L2-normalises it. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published, with its next-sentence-prediction head and pooler weights left unloaded. What the upstream checkpoint supplies is the encoder, the masked-LM head and the tokenizer; what the carried pipeline module adds is manifest verification, input validation and ceilings (over-long texts are rejected, not truncated), the two task methods, fixed output contracts and the `validate_inputs` and `evaluation_report` stage helpers. **The fill-mask `score` is not a calibrated probability** (it is a softmax ranking signal), and **embeddings are representations, not predictions**; the pipeline ships no threshold for either.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author a synthetic cloze sentence and a few sentences to embed (or upload your own), stage and digest-verify the immutable upstream snapshot, surface the pipeline's ceilings and validate both capabilities' inputs into one input manifest, run fill-mask and read its ranked candidates correctly, run embedding and read the vector contract correctly (shape, pooling, unit), read from the machine-readable evaluation report why no metric is reported and what labelled data each capability needs, and export identifiers alongside vectors plus provenance.

**This notebook does not demonstrate:** fine-tuning or classification heads, next-sentence prediction, multi-mask filling, raw-logit access, text generation (BERT is an encoder), cased or non-English text (the checkpoint is uncased English), or contrastively trained sentence similarity (the `qwen3-embedding-pipeline` sibling covers retrieval-grade embeddings). The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32; the pipeline loads the checkpoint in float32 on both). The model card's CPU smoke loaded and verified the snapshot in 4.34 s, filled one mask in 0.13 s and embedded two sentences in 0.02 s, so the default runs in seconds on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 440 MB `model.safetensors` are the largest downloads of the run.
- **Knowledge:** basic Python; what a softmax over a vocabulary is and why it is not a calibrated probability; what cosine similarity between unit vectors means.
- **Data:** the default sample is one synthetic cloze sentence and three synthetic sentences authored in code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 text file whose first non-empty line is a cloze sentence containing exactly one `[MASK]` and whose remaining non-empty lines are the sentences to embed. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google-bert/bert-base-uncased` snapshot (~441 MB) at revision `86b5e0934494…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'bert-masked-lm-pipeline',
    'repository_revision': 'db771a57d548a810ba84edaaff6a05cba6cf40aa',
    'embedded_module': 'src/bert_masked_lm_pipeline/pipeline.py',
    'module_sha256': 'b6c7d0e00a3f4c3e1855e00c679f8c515e6d858fc74b8e03e495328507840f4d',
    'generator': 'build_notebook.py/1',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/bert_masked_lm_pipeline/pipeline.py` @ `db771a57d548`)

This cell **is** the repository's pipeline module: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the module's, byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (currently 1: the default weights directory becomes working-directory-relative because a notebook has no `__file__`). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever this cell and the module diverge, so what you run here is what the repository tests. Nothing in this cell runs a model yet.

In [ ]:
"""Masked-language modelling and sentence embeddings over the pinned ``google-bert/bert-base-uncased``.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. Two task methods: ``fill_mask`` (one ``[MASK]`` token ->
ranked vocabulary candidates) and ``embed`` (CLS or mean pooled, L2-normalised 768-d representations).
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "google-bert/bert-base-uncased"
MODEL_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "bert-base-uncased"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. 512 is max_position_embeddings in the pinned config.json and model_max_length in
# tokenizer_config.json; longer inputs are rejected (not truncated) so a caller never silently loses [MASK].
MAX_TEXT_TOKENS = 512
MAX_TEXT_CHARS = 4_000  # pre-tokenisation guard; ~4 chars per WordPiece token on English text
MAX_BATCH = 64  # texts per embed() call
MAX_TOP_K = 100
VOCAB_SIZE = 30522  # config.json vocab_size
HIDDEN_SIZE = 768  # config.json hidden_size
MASK_TOKEN = "[MASK]"
POOLINGS = ("cls", "mean")
DEFAULT_TOP_K = 5


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_text(text: Any, name: str) -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return text


def _check_mask_count(text: str) -> str:
    """`fill_mask` accepts exactly one [MASK]; raise naming the count found."""
    if text.count(MASK_TOKEN) != 1:
        raise ValueError(f"text must contain exactly one {MASK_TOKEN}, found {text.count(MASK_TOKEN)}")
    return text


def _check_top_k(top_k: Any) -> int:
    if isinstance(top_k, bool) or not isinstance(top_k, int):
        raise TypeError("top_k must be an int")
    if not 1 <= top_k <= MAX_TOP_K:
        raise ValueError(f"top_k must be between 1 and MAX_TOP_K={MAX_TOP_K}")
    return top_k


def _check_batch(texts: Any, pooling: Any) -> list[str]:
    """`embed`'s batch contract; raise naming the first violated ceiling."""
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a list of str, not a single string")
    if not 1 <= len(texts) <= MAX_BATCH:
        raise ValueError(f"texts must hold 1..MAX_BATCH={MAX_BATCH} items, got {len(texts)}")
    clean = [_check_text(t, f"texts[{i}]") for i, t in enumerate(texts)]
    if pooling not in POOLINGS:
        raise ValueError(f"pooling must be one of {POOLINGS}")
    return clean


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "sequence of non-empty str; every entry carrying a [MASK] token is also a fill_mask input, "
        "every entry is an embed input (one vector per text)"
    ),
    "batch": [1, MAX_BATCH],
    "text_chars": [1, MAX_TEXT_CHARS],
    "text_tokens": [1, MAX_TEXT_TOKENS],
    "top_k": [1, MAX_TOP_K],
    "pooling": list(POOLINGS),
    "mask_token": MASK_TOKEN,
    "masks_per_fill_mask_text": 1,
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": HIDDEN_SIZE,
    "preprocessing": (
        "WordPiece tokenisation that lower-cases and strips accents; texts past MAX_TEXT_TOKENS are "
        "rejected, never truncated, so a [MASK] can never be silently lost; embed pools the last "
        "layer (cls position or attention-masked mean) and L2-normalises"
    ),
}


def validate_inputs(
    texts: Sequence[str],
    *,
    top_k: int = DEFAULT_TOP_K,
    pooling: str = "cls",
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    ``texts`` is the batch ``embed`` would take; every entry that carries a ``[MASK]`` token is
    additionally checked against ``fill_mask``'s contract (exactly one mask, ``top_k`` in range) and
    marked in the manifest. Both capabilities' checks run through the same private functions the
    methods use — ``_check_batch``/``_check_text`` for ``embed``, ``_check_mask_count``/``_check_top_k``
    for ``fill_mask`` — so a rejection here is a rejection there. ``MAX_TEXT_TOKENS`` is enforced
    after tokenisation inside the pipeline and therefore cannot be observed at this stage.
    """
    checked = _check_batch(texts, pooling)
    _check_top_k(top_k)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    inputs = []
    for i, text in enumerate(checked):
        masks = text.count(MASK_TOKEN)
        if masks:
            _check_mask_count(text)
        inputs.append(
            {
                "id": names[i] if names else f"text-{i}",
                "chars": len(text),
                "masks": masks,
                "fill_mask_input": bool(masks),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "top_k": top_k,
        "pooling": pooling,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], expected_tokens: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    Neither capability has a metric helper in this repository, so the verdict is always
    ``not-measurable`` (EVAL9). ``expected_tokens`` exists for interface parity with the fleet's
    other pipelines and is recorded in ``reason`` rather than scored: one author-expected token is
    an intent, not a labelled cloze set, and computing a hit rate from it would present a single
    observation as an accuracy. ``result`` is the ``fill_mask`` result; the embedding half is a
    representation and is covered by the same verdict.
    """
    candidates = result.get("candidates", [])
    supplied = expected_tokens is not None
    return {
        "task": "masked-language modelling (fill-mask) and sentence embedding",
        "score_semantics": (
            f"fill_mask `score` is a softmax over the {VOCAB_SIZE}-token vocabulary at the masked "
            "position — a ranking signal, not a calibrated probability, with argmax as the decision "
            f"rule and no shipped threshold; embed returns {HIDDEN_SIZE}-d unit vectors whose only "
            "meaning is cosine within the same model and pooling policy"
        ),
        "sample_kind": sample_kind,
        "n_candidates": len(candidates),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "the repository ships no metric helper for either capability"
            + (
                "; an expected token was supplied, but one author-expected token is an intent rather "
                "than a labelled cloze set, so scoring it would present a single observation as an accuracy"
                if supplied
                else "; the evaluated sample carries no gold tokens and no similarity labels"
            )
        ),
        "needs": (
            "for fill-mask, a labelled cloze set (sentence, mask position, gold token) over enough "
            "sentences to state a dispersion, scored with the caller's own top-1/top-k hit-rate code; "
            "for the embeddings, a judged similarity set (Spearman correlation) or a retrieval or "
            "clustering set with relevance labels (recall@k) — neither of which this repository ships"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class BERTMaskedLMPipeline:
    """``_mask_runner`` maps one text to (vocab logits at the [MASK] position, n_tokens);
    ``_embed_runner`` maps texts to (last hidden states (N, T, 768), attention mask (N, T)); both injectable.
    ``_decode`` maps a token id to its string."""

    _mask_runner: Callable[[str], tuple[np.ndarray, int]]
    _embed_runner: Callable[[list[str]], tuple[np.ndarray, np.ndarray]]
    _decode: Callable[[int], str]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BERTMaskedLMPipeline:
        import torch
        from transformers import AutoTokenizer, BertForMaskedLM

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
            origin = "local-snapshot"
        elif allow_download:
            source, kwargs, origin = MODEL_ID, {}, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        tokenizer = AutoTokenizer.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = BertForMaskedLM.from_pretrained(
            source, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        mask_id = tokenizer.mask_token_id

        def mask_runner(text: str) -> tuple[np.ndarray, int]:
            batch = tokenizer(text, return_tensors="pt", truncation=False).to(resolved_device)
            n_tokens = int(batch["input_ids"].shape[1])
            if n_tokens > MAX_TEXT_TOKENS:
                raise ValueError(f"text tokenises to {n_tokens} tokens; MAX_TEXT_TOKENS={MAX_TEXT_TOKENS}")
            position = (batch["input_ids"][0] == mask_id).nonzero().flatten()
            with torch.inference_mode():
                logits = model(**batch).logits[0, position[0]]
            return logits.float().cpu().numpy(), n_tokens

        def embed_runner(texts: list[str]) -> tuple[np.ndarray, np.ndarray]:
            batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=False)
            if batch["input_ids"].shape[1] > MAX_TEXT_TOKENS:
                raise ValueError(f"a text tokenises past MAX_TEXT_TOKENS={MAX_TEXT_TOKENS}")
            batch = batch.to(resolved_device)
            with torch.inference_mode():
                hidden = model.bert(**batch).last_hidden_state
            return hidden.float().cpu().numpy(), batch["attention_mask"].cpu().numpy()

        return cls(mask_runner, embed_runner, tokenizer.convert_ids_to_tokens, resolved_device, origin)

    def fill_mask(self, text: str, top_k: int = DEFAULT_TOP_K) -> dict[str, Any]:
        """Rank candidates for exactly one ``[MASK]``; ``score`` is a softmax over the 30 522-token vocab."""
        text = _check_mask_count(_check_text(text, "text"))
        top_k = _check_top_k(top_k)
        logits, n_tokens = self._mask_runner(text)
        logits = np.asarray(logits, dtype=np.float64)
        if logits.shape != (VOCAB_SIZE,):
            raise RuntimeError(f"backend returned {logits.shape}, expected ({VOCAB_SIZE},)")
        shifted = np.exp(logits - logits.max())
        probs = shifted / shifted.sum()
        order = np.argsort(-probs, kind="stable")[:top_k]
        candidates = [
            {
                "token": self._decode(int(i)),
                "token_id": int(i),
                "score": float(probs[i]),
                "sequence": text.replace(MASK_TOKEN, self._decode(int(i)), 1),
            }
            for i in order
        ]
        return {
            "candidates": candidates,
            "top_k": top_k,
            "n_tokens": n_tokens,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def embed(self, texts: Sequence[str], pooling: str = "cls") -> dict[str, Any]:
        """L2-normalised 768-d representations: CLS token or attention-masked mean of the last layer."""
        clean = _check_batch(texts, pooling)
        hidden, mask = self._embed_runner(clean)
        hidden = np.asarray(hidden, dtype=np.float32)
        mask = np.asarray(mask, dtype=np.float32)
        if hidden.ndim != 3 or hidden.shape[0] != len(clean) or hidden.shape[2] != HIDDEN_SIZE:
            raise RuntimeError(f"backend returned {hidden.shape}, expected ({len(clean)}, T, {HIDDEN_SIZE})")
        if pooling == "cls":
            pooled = hidden[:, 0]
        else:
            counts = np.maximum(mask.sum(axis=1, keepdims=True), 1.0)
            pooled = (hidden * mask[:, :, None]).sum(axis=1) / counts
        normalized = pooled / np.maximum(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12)
        return {
            "embeddings": normalized.tolist(),
            "dim": HIDDEN_SIZE,
            "pooling": pooling,
            "normalized": True,
            "n_tokens": [int(v) for v in mask.sum(axis=1)],
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `86b5e0934494…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BERTMaskedLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bert-base-uncased",
  "modelId": "google-bert/bert-base-uncased",
  "revision": "86b5e0934494bd15c9632b12f734a8a67f723594",
  "files": [
    {
      "path": "LICENSE",
      "bytes": 11356,
      "sha256": "43070e2d4e532684de521b885f385d0841030efa2b1a20bafb76133a5e1379c1"
    },
    {
      "path": "README.md",
      "bytes": 10517,
      "sha256": "9187b6018ea0010d884e78e098e328faa1b88b301570d0cce606bb35e4067e17"
    },
    {
      "path": "config.json",
      "bytes": 570,
      "sha256": "7160e1553ad2ca51d8c1cb066be533db31826e12d173824c1bb0cb1a4f187d20"
    },
    {
      "path": "coreml/fill-mask/float32_model.mlpackage/Manifest.json",
      "bytes": 617,
      "sha256": "4e9566ed44c3c401dada9ff243e0efbd570f9bc4e55dd7fdfb937cb23993c94c"
    },
    {
      "path": "model.safetensors",
      "bytes": 440449768,
      "sha256": "68d45e234eb4a928074dfd868cead0219ab85354cc53d20e772753c6bb9169d3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 466062,
      "sha256": "ce64fce797c24f68df90b40a3f74f579b336a493db14bd583fd520ea0d8c9a98"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 48,
      "sha256": "a025160ef0431f1a392f6f050c1310f4c5d9fb6f275932dbccba73c4d214bf10"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 441170446
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print({'verified_files': [entry['path'] for entry in snapshot.get('files', [])]})
pipe = BERTMaskedLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**, written in this cell: one cloze sentence with a single `[MASK]` (the model card's smoke sentence) together with the token its author expects, and three sentences to embed — two about a pet on a floor covering and one unrelated — each given a stable identifier (`s1`, `s2`, `s3`) so every vector can be mapped back to its text. The expected token is the author's intent, not a labelled dataset: whether it lands in the top-`k` is a smoke/sanity check that the code path works, never an accuracy figure and never benchmark evidence; the three sentences carry **no similarity labels**, so the cosine table they produce is a qualitative check only. `TOP_K` and `POOLING` are Colab form parameters checked against the carried module in Section 5.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file whose first non-empty line is a cloze sentence containing exactly one `[MASK]` and whose remaining non-empty lines (at most `MAX_BATCH`) are the sentences to embed, each at most `MAX_TEXT_CHARS` characters and at most `MAX_TEXT_TOKENS` WordPiece tokens (longer texts are rejected by the pipeline, not truncated). Text is lower-cased by the tokenizer, so case carries no information. The upload stays inside this runtime. If you also hold gold tokens or similarity judgements, keep them outside the notebook — Section 8 explains what to compute with them.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
TOP_K = 5  # @param {type:"integer"}
POOLING = 'mean'  # @param ["cls", "mean"]

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    lines = [line.strip() for line in io.StringIO(uploaded[sample_name].decode('utf-8')) if line.strip()]
    if len(lines) < 2:
        raise ValueError(f'{sample_name}: expected a [MASK] sentence line followed by at least one sentence line')
    cloze, sentences = lines[0], lines[1:]
    expected_token = None
    sample_kind = 'BYOD upload'
else:
    cloze = 'The capital of France is [MASK].'
    expected_token = 'paris'
    sentences = [
        'The cat sat on the mat.',
        'A dog lay on the rug.',
        'Interest rates were raised by a quarter of a point.',
    ]
    sample_name = 'synthetic_cloze_and_sentences'
    sample_kind = 'synthetic (authored in this cell; the model card smoke cloze sentence)'
sentence_ids = [f's{index + 1}' for index in range(len(sentences))]
sample_sha256 = hashlib.sha256('\n'.join([cloze, *sentences]).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'cloze': cloze, 'expected_token': expected_token, 'sentences': len(sentences), 'top_k': TOP_K, 'pooling': POOLING, 'text_sha256': sample_sha256})
for sentence_id, sentence in zip(sentence_ids, sentences, strict=True):
    print(f'{sentence_id}: {sentence[:100]}')

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage and covers **both** capabilities: it takes the batch `embed` would take, and every entry carrying a `[MASK]` token is additionally checked against `fill_mask`'s contract and marked `fill_mask_input` in the manifest. The checks are the methods' own — `_check_batch`/`_check_text` for `embed`, `_check_mask_count`/`_check_top_k` for `fill_mask` — so a rejection here is a rejection there. `MAX_TEXT_CHARS` is the character guard applied before tokenisation; `MAX_TEXT_TOKENS` (512, the checkpoint's position limit) is applied after tokenisation and **rejects** longer texts rather than truncating them, so it is enforced inside the pipeline and cannot be observed at this stage; `MAX_BATCH` bounds one `embed` call; `MAX_TOP_K` bounds `top_k`; `MASK_TOKEN` must occur exactly once in a fill-mask text; `POOLINGS` names the two pooling policies; `VOCAB_SIZE` and `HIDDEN_SIZE` are the output-vocabulary and vector widths the contracts promise. The manifest is written to `outputs/bert_masked_lm_input_manifest.json`. To show what rejection looks like, the cell also validates a two-mask text and records the pipeline's own error message as a finding. The notebook never trims or alters the texts.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_BATCH': MAX_BATCH, 'MAX_TOP_K': MAX_TOP_K, 'VOCAB_SIZE': VOCAB_SIZE, 'HIDDEN_SIZE': HIDDEN_SIZE, 'MASK_TOKEN': MASK_TOKEN, 'POOLINGS': POOLINGS, 'DEFAULT_TOP_K': DEFAULT_TOP_K}
print(ceilings)
input_manifest = validate_inputs([cloze, *sentences], top_k=TOP_K, pooling=POOLING, names=['cloze', *sentence_ids])
# Demonstrate the exactly-one-mask rejection; the finding is recorded, not swallowed.
try:
    validate_inputs([f'A {MASK_TOKEN} and another {MASK_TOKEN}.'], top_k=TOP_K, pooling=POOLING)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'two-mask-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/bert_masked_lm_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
print({'token_ceiling': f'MAX_TEXT_TOKENS={MAX_TEXT_TOKENS} is checked by the pipeline after tokenisation and rejects, never truncates'})

## 6. Fill the mask and read the candidates correctly

**Input/output contract.** `fill_mask(text, top_k=...)` takes one string with exactly one `[MASK]` and returns `candidates` — a list of `top_k` entries ordered by descending `score`, each with the WordPiece `token`, its `token_id`, the `score`, and the `sequence` with the mask substituted — plus `top_k`, `n_tokens` (WordPiece count including `[CLS]`/`[SEP]`), the device and the model identity. **Score semantics:** `score` is the softmax over the 30,522-token vocabulary at the masked position — a ranking signal that sums to 1 over the whole vocabulary, not a calibrated probability that the candidate is correct; the **default decision rule is `argmax`** (the first candidate) and the pipeline applies no minimum score, so a nonsensical sentence still yields a ranked list. Any acceptance threshold is owned by the caller and must be set on their own labelled cloze data. The model card's CPU smoke on this sentence ranked `paris` first with score 0.4168 (9 tokens); that is one observation, not an expected value — near-tied candidates can reorder between CPU and CUDA kernels. Inference is deterministic on a fixed device and dtype (`model.eval()`, no sampling, no seed needed).

In [ ]:
import time

started = time.perf_counter()
filled = pipe.fill_mask(cloze, top_k=TOP_K)
fill_elapsed = time.perf_counter() - started
scores = [candidate['score'] for candidate in filled['candidates']]
fill_checks = {
    'top_k_candidates_returned': len(filled['candidates']) == TOP_K,
    'scores_descending': all(a >= b for a, b in zip(scores, scores[1:], strict=False)),
    'scores_in_unit_interval': all(0.0 < s <= 1.0 for s in scores),
    'tokens_within_vocab': all(0 <= candidate['token_id'] < VOCAB_SIZE for candidate in filled['candidates']),
    'n_tokens_within_ceiling': 1 <= filled['n_tokens'] <= MAX_TEXT_TOKENS,
}
if not all(fill_checks.values()):
    raise RuntimeError(f'fill_mask output failed a sanity check: {fill_checks}')
print({key: value for key, value in filled.items() if key != 'candidates'})
print({'seconds': round(fill_elapsed, 3), 'checks': fill_checks, 'decision_rule': 'argmax = first candidate; no threshold shipped'})
print(f'cloze: {cloze}')
for rank, candidate in enumerate(filled['candidates'], start=1):
    print(f"{rank:>2}. {candidate['token']:<12} id {candidate['token_id']:>6}  score {candidate['score']:.4f}  {candidate['sequence']}")
fill_sanity = {}
if expected_token is not None:
    fill_sanity = {'expected_token_in_top_k': expected_token in [candidate['token'] for candidate in filled['candidates']]}
    print({'sanity_check': fill_sanity, 'note': 'falsifiable plumbing check on one synthetic sentence; not a metric'})

## 7. Embed the sentences and read the vectors correctly

**Input/output contract.** `embed(texts, pooling=...)` takes a list of 1..`MAX_BATCH` strings and returns `embeddings` — one list per input text, **in input order**, each of length `dim` (768) — plus `pooling` (`cls`: the `[CLS]` position of the last layer; `mean`: the attention-masked mean of the last layer, so padding never enters the average), `normalized` (`True`: every vector has unit L2 norm), `n_tokens` per text, the device and the model identity. The unit of embedding is **one vector per text**; there is no per-token or per-chunk output, and a text over `MAX_TEXT_TOKENS` is rejected rather than chunked or truncated. Missing data has no meaning here: empty strings are rejected, not embedded. **Embeddings are representations, not predictions:** they carry no label and no confidence, and their only meaning is relative — cosine between two vectors from the same model and the same pooling policy.

**No intrinsic metric exists** for an embedding: the repository ships no metric helper and no labelled data, and quality can only be judged through a downstream task with labels — a similarity benchmark with human judgements (Spearman correlation), or a retrieval or clustering set with relevance labels (recall@k). Below, the pairwise cosine matrix is computed from the returned vectors as a **qualitative check** that the contract works: on the default sample the two pet sentences are expected to score higher with each other than with the unrelated one. Cosine values are similarities in `[-1, 1]` on this model's geometry; raw BERT was not trained with a sentence-similarity objective, so its cosine scale is compressed and uncalibrated (the model card's smoke gave 0.8719 for the two pet sentences with mean pooling — one observation, not an expected value), absolute values are not comparable across models or pooling policies, and any "same meaning" threshold is the caller's to set on labelled pairs. Look for `(N, 768)` unit-norm vectors, token counts within the ceiling, and no truncation (the pipeline cannot truncate).

In [ ]:
started = time.perf_counter()
embedding_result = pipe.embed(sentences, pooling=POOLING)
embed_elapsed = time.perf_counter() - started
vectors = np.asarray(embedding_result['embeddings'], dtype=np.float32)
norms = np.linalg.norm(vectors, axis=1)
embed_checks = {
    'one_vector_per_text': vectors.shape == (len(sentences), HIDDEN_SIZE),
    'dim_matches_contract': embedding_result['dim'] == HIDDEN_SIZE,
    'pooling_as_requested': embedding_result['pooling'] == POOLING,
    'unit_norm': bool(embedding_result['normalized']) and bool(np.allclose(norms, 1.0, atol=1e-4)),
    'all_values_finite': bool(np.isfinite(vectors).all()),
    'n_tokens_within_ceiling': all(1 <= n <= MAX_TEXT_TOKENS for n in embedding_result['n_tokens']),
}
if not all(embed_checks.values()):
    raise RuntimeError(f'embed output failed a sanity check: {embed_checks}')
print({key: value for key, value in embedding_result.items() if key != 'embeddings'})
print({'seconds': round(embed_elapsed, 3), 'shape': vectors.shape, 'norms': [round(float(n), 4) for n in norms], 'unit': 'one vector per text', 'checks': embed_checks})
cosine = vectors @ vectors.T
for sentence_id, row in zip(sentence_ids, cosine, strict=True):
    print(sentence_id, {other_id: round(float(value), 4) for other_id, value in zip(sentence_ids, row, strict=True)})
embed_sanity = {}
if not USE_BYOD:
    embed_sanity = {'related_pair_outscores_unrelated': bool(cosine[0, 1] > max(cosine[0, 2], cosine[1, 2]))}
    print({'sanity_check': embed_sanity, 'note': 'qualitative check on three synthetic sentences; not a metric'})

## 8. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — even, as here, when nothing is measurable. The repository ships **no metric helper and reports no performance measure** for either capability, so the verdict is always `not-measurable` and the report states what would make each task measurable: for fill-mask, a labelled cloze set (sentence, mask position, gold token) over enough sentences to state a dispersion, scored with the caller's own top-1/top-k hit-rate code; for the embeddings, a judged similarity set (Spearman correlation) or a retrieval or clustering set with relevance labels (recall@k). Supplying the author's expected token does **not** change the verdict — one expected token is an intent, not a labelled cloze set, and scoring it would present a single observation as an accuracy — so the helper records that in `reason` instead. The sanity checks printed in Sections 6 and 7 remain falsifiable plumbing checks, not results. The report is written to `outputs/bert_masked_lm_evaluation_report.json`.

In [ ]:
expected_tokens = None if expected_token is None else [expected_token]
report = evaluation_report(filled, expected_tokens, sample_kind=sample_kind)
with open('outputs/bert_masked_lm_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is reported: neither capability has a metric helper here; compute top-k hit rates on your own labelled cloze set and Spearman or recall@k on your own judged pairs.')

## 9. Export identifiers alongside vectors, and provenance

Two further files are written under `outputs/` beside the input manifest and the evaluation report. The vectors go to CSV (`outputs/bert_masked_lm_embeddings.csv`) with one row per sentence — `id`, `n_tokens`, `pooling`, then `e0000…e0767` — so every vector stays attached to its identifier for downstream use. One JSON record (`outputs/bert_masked_lm_result.json`) preserves the fill-mask result (the cloze, every candidate with token, id, score, rank and sequence, the decision rule, `n_tokens`), the embedding contract (`dim`, `pooling`, `normalized`, unit), the identified sentences with their token counts, the cosine table keyed by identifier, the sanity checks, the ceilings in force, the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

with open('outputs/bert_masked_lm_embeddings.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['id', 'n_tokens', 'pooling'] + [f'e{i:04d}' for i in range(HIDDEN_SIZE)])
    for sentence_id, vector, n_tokens in zip(sentence_ids, embedding_result['embeddings'], embedding_result['n_tokens'], strict=True):
        writer.writerow([sentence_id, n_tokens, embedding_result['pooling']] + [f'{value:.7f}' for value in vector])
payload = {
    'fill_mask': {
        'cloze': cloze,
        'expected_token': expected_token,
        'candidates': [{'rank': rank, **candidate} for rank, candidate in enumerate(filled['candidates'], start=1)],
        'decision_rule': 'argmax (first candidate); score is a vocabulary softmax, not a calibrated probability; no threshold shipped',
        'top_k': filled['top_k'],
        'n_tokens': filled['n_tokens'],
        'seconds': round(fill_elapsed, 3),
        'sanity_checks': fill_checks,
        'plumbing_check': fill_sanity,
    },
    'embed': {
        'contract': {'dim': embedding_result['dim'], 'pooling': embedding_result['pooling'], 'normalized': embedding_result['normalized'], 'unit': 'one vector per text'},
        'texts': dict(zip(sentence_ids, sentences, strict=True)),
        'n_tokens': dict(zip(sentence_ids, embedding_result['n_tokens'], strict=True)),
        'cosine_similarity': {sentence_id: {other_id: float(value) for other_id, value in zip(sentence_ids, row, strict=True)} for sentence_id, row in zip(sentence_ids, cosine, strict=True)},
        'vectors_file': 'outputs/bert_masked_lm_embeddings.csv',
        'seconds': round(embed_elapsed, 3),
        'sanity_checks': embed_checks,
        'plumbing_check': embed_sanity,
    },
    'ceilings': ceilings,
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'text_sha256': sample_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/bert_masked_lm_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The fill-mask candidates are a softmax ranking over the vocabulary at one masked position: the first entry is the argmax, the `score` is not a calibrated probability, and the pipeline applies no threshold — the caller owns any cut-off and must set it on labelled cloze data. The embeddings are unit vectors in this model's 768-dimensional space: representations that predict nothing, whose only meaning is cosine between vectors from the same model and pooling policy; raw BERT's cosine scale is compressed and uncalibrated, and a retrieval-grade sentence encoder is a different model. On the synthetic sample both outputs are plumbing evidence only; the evaluation report is `not-measurable` because no metric can be computed without labelled data, and a real evaluation needs a labelled cloze set (top-k hit rates) and a judged similarity or retrieval set (Spearman, recall@k) with the caller's own code over enough items to state a dispersion. Text is lower-cased and accent-stripped by the tokenizer; texts over 512 WordPiece tokens are rejected, not truncated; the checkpoint is uncased English only and carries the gender and occupation associations the upstream card documents; the pipeline exposes no fine-tuning, generation, multi-mask filling or next-sentence prediction. Inference is deterministic on a fixed device and dtype, but CPU and CUDA kernels can reorder near-tied candidates and perturb low-order embedding digits.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated inputs against the enforced ceilings, execute both public pipeline paths, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, cloze accuracy or similarity quality on any domain, a usable threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments.** Switch `POOLING` between `cls` and `mean` and watch the cosine table move — the same sentences, a different representation; write a cloze sentence whose answer is ambiguous and read how the softmax mass spreads across candidates; assemble a dozen labelled cloze sentences of your own and compute the top-1 and top-5 hit rates the evaluation report asks for; judge a handful of sentence pairs yourself and compute Spearman correlation against the cosine values. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google-bert/bert-base-uncased
- Upstream code: https://github.com/google-research/bert
- BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding: https://arxiv.org/abs/1810.04805